In [10]:
import os
os.listdir("/content")

['.config', 'flyrank-ml-internship', 'sample_data']

In [11]:
!git clone https://github.com/gitWithPrince272/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 148 (delta 60), reused 96 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.86 MiB | 9.89 MiB/s, done.
Resolving deltas: 100% (60/60), done.


In [12]:
import os
os.chdir("/content/flyrank-ml-internship")
print(os.getcwd())

/content/flyrank-ml-internship


In [13]:
import pandas as pd
import glob

csv_files = glob.glob("data/raw/*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0])
df.head()

['data/raw/content_refresh_anonymized.csv']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gitWithPrince272/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked Actions

### High Priority
- Pages with high impressions, low CTR, and average position between 5 and 20 should be optimized first because they have the highest opportunity to gain more clicks.

### Medium Priority
- Pages with moderate impressions or CTR should be reviewed after the high-priority pages.

### Low Priority
- Pages with very low impressions or already ranking in the top positions require lower priority.

### Reason Codes
- **CONFIRMED** – High impressions, low CTR, and position between 5–20.
- **MIXED** – Meets some conditions but not all.
- **OPPOSITE** – Low impressions or already ranks very well.
- **FALSE** – Does not meet the rule or has insufficient data.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Create reason codes
df["reason_code"] = "FALSE"

confirmed = (
    (df["impressions_90d"] > df["impressions_90d"].median()) &
    (df["ctr"] < df["ctr"].median()) &
    (df["avg_position"].between(5, 20))
)

mixed = (
    (df["impressions_90d"] > df["impressions_90d"].median()) |
    (df["ctr"] < df["ctr"].median())
)

opposite = (
    (df["impressions_90d"] <= df["impressions_90d"].median()) |
    (df["avg_position"] < 5)
)

df.loc[confirmed, "reason_code"] = "CONFIRMED"
df.loc[mixed & ~confirmed, "reason_code"] = "MIXED"
df.loc[opposite & ~(confirmed | mixed), "reason_code"] = "OPPOSITE"

display(
    df[
        [
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "reason_code",
        ]
    ].head(10)
)


,content_id,impressions_90d,ctr,avg_position,reason_code
0,content_304f48230142,3803,0.76,10.6,MIXED
1,content_a1fb4e703a9e,15320,0.05,20.3,MIXED
2,content_9aa793d4d895,12581,0.09,36.5,MIXED
3,content_331d6c4de07b,11751,0.49,6.2,MIXED
4,content_d99b7a2d90ca,19140,0.13,44.0,MIXED
5,content_d4084a4bc775,3970,0.03,8.5,CONFIRMED
6,content_9a34b442b552,20,0.00,7.0,MIXED
7,content_a63219c6e95a,1724,0.06,21.2,MIXED
8,content_5e6c160719bc,32574,0.09,46.0,MIXED
9,content_c27558df2b0c,1240,0.16,4.9,MIXED


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended Use and Limits

### Intended Use
- This playbook helps prioritize SEO pages for manual review.
- It is intended for SEO analysts and content teams.
- The recommendations support decision-making and should not replace human judgment.

### Limits
- The model uses historical data only.
- It cannot predict future search trends or algorithm updates.
- Final decisions should always be reviewed by a human.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended user: SEO analysts and content teams")
print("Purpose: Prioritize pages for manual SEO review")
print("Limit 1: Uses historical data only")
print("Limit 2: Human review is required before action")

Intended user: SEO analysts and content teams
Purpose: Prioritize pages for manual SEO review
Limit 1: Uses historical data only
Limit 2: Human review is required before action


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human Review and No-Go List

### Human Review
- Verify that the recommendation matches business goals.
- Confirm that the page content is accurate and up to date.
- Review search intent before making changes.

### No-Go List
- Do not automatically publish content changes.
- Do not remove high-performing pages without review.
- Do not rely only on model predictions for business decisions.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Human review checklist:")
print("- Verify business goals")
print("- Review page quality")
print("- Confirm search intent")

print("\nNo-Go actions:")
print("- No automatic publishing")
print("- No deletion of high-performing pages")
print("- No decisions based only on the model")


Human review checklist:
- Verify business goals
- Review page quality
- Confirm search intent

No-Go actions:
- No automatic publishing
- No deletion of high-performing pages
- No decisions based only on the model


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring and Retrain Triggers

### Monitor
- Track changes in impressions, CTR, and average position.
- Monitor changes in search trends and business priorities.
- Review model performance regularly.

### Retrain Triggers
- Significant drop in CTR or traffic.
- Large changes in search behavior or algorithm updates.
- Availability of new historical data.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Monitoring:")
print("- Track impressions, CTR, and average position")
print("- Monitor search trends")
print("- Review model performance")

print("\nRetrain when:")
print("- CTR or traffic drops significantly")
print("- Search behavior changes")
print("- New historical data becomes available")



Monitoring:
- Track impressions, CTR, and average position
- Monitor search trends
- Review model performance

Retrain when:
- CTR or traffic drops significantly
- Search behavior changes
- New historical data becomes available


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports for the Paper

The ranked action queue is exported for documentation and future review.

The exported file can be reused in reports or presentations and supports the recommendations made by the playbook.

The export contains the content ID, ranking information, and reason codes for each recommended page.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

export_cols = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "reason_code"
]

df[export_cols].to_csv(
    "work/outputs/action_playbook.csv",
    index=False
)

print("Export completed successfully!")
print("Saved to: work/outputs/action_playbook.csv")


Export completed successfully!
Saved to: work/outputs/action_playbook.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- Completed all five sections of the action playbook.
- Exported the ranked action queue to `work/outputs/action_playbook.csv`.
- Used careful, evidence-based recommendations.
- Included human review and monitoring guidance.
- Ready for GitHub submission.